# Fine-tuning Vision Transformers: frozen head, LLRD, and LoRA

How to adapt a pretrained ViT backbone (DINOv2/DINOv3, RF-DETR's DINOv2 encoder, timm ViTs) to a new
task, and how to choose between the three strategies that actually get used in practice.

Related notebooks: [Vision Transformer](../transformer/vit.ipynb) ·
[DINOv3](../foundation_models/DINOv3.ipynb) ·
[Transfer learning, backbone/neck/head](transfer_learning_fine_tuning.ipynb) ·
[RAG](../rag/index.ipynb)

Runnable companion script: [`scripts/vit_ft_examples.py`](scripts/vit_ft_examples.py) — all three
strategies on the same DINOv3 ViT-B/16, one training step each.


## 1. Why ViTs are fine-tuned differently from CNNs

A Vision Transformer has no convolutions: it cuts the image into 16 × 16 px patches, turns each patch
into a token, and runs a stack of identical transformer blocks over all tokens at once. Two consequences
follow for fine-tuning.

1. **The pretrained features are already very general.** DINOv2/DINOv3 were trained self-supervised on
   over a billion images. Their features transfer to new tasks *without being changed*, so a frozen
   backbone is a real option, not just a fallback for when you are short of GPU memory.
2. **The backbone is fragile under a large learning rate.** A ViT has no convolutional prior to fall
   back on. Update every layer at the head's learning rate and the early layers drift away from what
   made them good — catastrophic forgetting. So when you *do* train the backbone, the early layers get
   a much smaller learning rate.

A CNN such as ResNet or YOLO is sturdier here, which is why its usual recipe — train everything, maybe
freeze the first few blocks — works fine and needs no special treatment.


## 2. The three strategies

| Strategy | What trains | GPU memory | When it fits |
| --- | --- | --- | --- |
| **Frozen backbone + head** (linear probe or small decoder) | only the new head; the backbone runs under `torch.no_grad()` | lowest: no backbone gradients, no optimiser state for them | little data, strong foundation model, fast iteration |
| **Full fine-tune with layer-wise LR decay (LLRD)** | everything; each backbone layer gets a smaller LR the closer it is to the input | highest: gradients + AdamW state for every weight | enough data (tens of thousands of images), task far from pretraining |
| **Parameter-efficient (LoRA, adapters)** | small low-rank matrices beside the attention projections; original weights frozen | low | very large backbones, or many tasks sharing one backbone |

**Rule of thumb.** Start frozen and measure. Move to LLRD only if the frozen model plateaus *and* you
have the data. Reach for LoRA when the full fine-tune does not fit in memory.

Memory, roughly, for AdamW in mixed precision: a trainable parameter costs its own weight plus a
gradient plus two optimiser moments. Freezing the backbone removes all three for 86 M of the 87 M
parameters of a ViT-B, which is why batch 8 at 896 px fits on a 24 GB card while the full fine-tune of
the same model at the same resolution does not.


## 3. Strategy A — frozen backbone, trained head

### 3.1 Architecture

```mermaid
flowchart LR
  A[RGB image<br/>896 px long side] --> B[ViT-B/16<br/>frozen, bf16]
  B --> C[Hidden states of<br/>blocks 3, 6, 9, 12]
  C --> D[DPT-style head<br/>trained]
  A --> E[RGB detail stem<br/>1/4 resolution]
  E --> D
  D --> F[One sigmoid map<br/>per class]
```

### 3.2 The three lines that do the freezing

```python
self.backbone.eval().requires_grad_(False)      # 1. no gradients, dropout/DropPath off

def train(self, mode=True):                     # 2. keep it in eval when the model is set to train
    super().train(mode)
    if self.freeze:
        self.backbone.eval()
    return self

with torch.no_grad():                           # 3. no autograd graph, no stored activations
    hs = self.backbone(pixel_values=x, output_hidden_states=True).hidden_states
```

Each of them matters:

* Forget `requires_grad_(False)` and the optimiser happily trains the backbone at the head's LR.
* Forget the `train()` override and a later `model.train()` silently switches the backbone's dropout
  and stochastic depth back on, so the "frozen" features become noisy between epochs.
* Forget `torch.no_grad()` and PyTorch keeps every backbone activation for a backward pass that never
  comes — that alone can eat most of your VRAM.

### 3.3 Which layers to tap

A ViT-B has 12 blocks. Take the hidden states after blocks `[n//4, n//2, 3n//4, n]` = 3, 6, 9, 12,
apply the final `LayerNorm`, and reshape each token list back into a 2-D grid (896 px / 16 = 56 × 56
tokens). Early blocks carry edges and texture; late blocks carry semantics. A dense-prediction decoder
needs both.

Token layout matters: a DINOv3 output is `[CLS] + register tokens + one token per patch` in row-major
order. Drop the first `1 + num_register_tokens` before reshaping, or every feature map is shifted.

```python
skip = 1 + cfg.num_register_tokens                       # CLS + 4 registers
h, w = H // patch, W // patch
feat = norm(hidden)[:, skip:].transpose(1, 2).reshape(B, -1, h, w)
```

### 3.4 The head (DPT-lite) and the detail stem

Each of the four maps gets a 1 × 1 conv down to 128 channels and is resampled to its own scale
(×4 up, ×2 up, as-is, ×2 down → 1/4 … 1/32 of the image); a top-down path fuses them back to 1/4 with
residual conv blocks. This is the [DPT](https://arxiv.org/abs/2103.13413) design, cut down.

Add a **detail stem**: a ViT only sees 16 px patches, but a hairline crack is 1–3 px wide. A few plain
convs over the raw RGB at 1/4 resolution, concatenated with the fused features before the output layer,
recover the thin structures. This is the part a CNN gives you for free.

Output one **sigmoid** channel per class rather than a softmax when classes can co-occur (a pixel can be
both *crack* and *stain*).

### 3.5 Training settings that worked

| Setting | Value | Why |
| --- | --- | --- |
| Input | long side 896 px, padded to a multiple of 16, random crop | patch size 16 means the sides must divide by 16 |
| Augmentation | scale 0.7–1.15, h-flip, mild brightness/contrast/saturation jitter | kept light: the features are fixed, so heavy augmentation mostly adds noise |
| Optimiser | AdamW on the **head only**, lr 1e-3, wd 1e-4 | a high LR is safe because every trainable weight is freshly initialised |
| Schedule | 300-iteration linear warm-up, then cosine to 2 % | |
| Loss | BCE with a per-class positive weight + soft Dice | thin classes cover < 1 % of pixels; plain BCE learns "never crack" |
| Precision | bf16 autocast | fp16 can overflow inside attention; fp32 doubles the memory |
| Grad clipping | norm 1.0 | |
| Checkpoint pick | best **validation** mean IoU; holdout never touched during training | |

### 3.6 Partial labels — the trick that matters most on merged datasets

When you merge several public datasets, most of them annotate only one or two of your classes. A rust
dataset never marks cracks, so an unmarked crack there is **not** a negative example.

Mask the loss: the term for class *c* on an image counts only if that image's source dataset annotates
*c* at all. Without the mask the model learns to ignore cracks on every rust photo.

```python
# valid: (B, C) bool — does this image's source dataset annotate this class?
loss = (per_class_loss * valid).sum() / valid.sum().clamp(min=1)
```

### 3.7 What the frozen ceiling looks like

In one defect-segmentation project a frozen DINOv3 ViT-B/16 with this head reached **0.50 mean IoU** on
validation after 3.2 h on a single RTX 3090 — cheap and quick, but *uneven per class*: it found 69.1 %
of cracked photos at 12.6 % false alarms, while on another class it fired on 72.1 % of photos that
contained nothing at all.

That is the usual frozen-backbone ceiling: the frozen features cannot learn a new notion of
"not a defect"; only the small head can, and a small head is not enough when the negative class is what
has to be learned.


## 4. Strategy B — full fine-tune with layer-wise LR decay (LLRD)

Everything trains: backbone, projector, decoder, head. The backbone is protected not by freezing but by
giving each layer a smaller learning rate the closer it sits to the input.

### 4.1 The rule

```latex
\mathrm{lr}_{\text{layer}} = \mathrm{lr}_{\text{encoder}} \times d_{\text{layer}}^{\,(L+1-\text{layer\_id})} \times d_{\text{component}}^{\,2}
```

`layer_id` is 0 for the patch embeddings and 1 … L for the transformer blocks; `d_layer` ≈ 0.8 is the
per-layer decay; the optional `d_component` term (≈ 0.7) further slows whole sub-modules. With
`lr_encoder = 1.5e-4`, `d_layer = 0.8`, `L = 13`, this gives:

| Part of the network | `layer_id` | Learning rate |
| --- | --- | --- |
| Decoder, projector, mask head | not decayed | 1.0e-4 (`lr`) |
| Backbone, last block | 12 | 4.7e-5 |
| Backbone, middle block | 6 | 1.2e-5 |
| Backbone, first block | 1 | 4.0e-6 |
| Patch embeddings | 0 | 3.2e-6 |

The patch embeddings move ~30× more slowly than the decoder: early layers barely change, late layers
adapt to the new task. This is the standard ViT recipe from
[BEiT](https://arxiv.org/abs/2106.08254) and [MAE](https://arxiv.org/abs/2111.06377), and it is what
`rfdetr` does internally (`get_dinov2_lr_decay_rate` in its backbone module).

Two details that are easy to get wrong:

* The backbone's *base* LR should sit below the head's — decay alone is not enough.
* Norms, biases and special tokens (CLS, registers, position embeddings) get **no weight decay**.
* A scheduler scales every param group by the same factor, so the ratios survive — you can use cosine
  on top of LLRD without recomputing anything.

### 4.2 A detection/segmentation example (RF-DETR-Seg)

```mermaid
flowchart LR
  A[Image 432 px] --> B[DINOv2 ViT backbone<br/>windowed attention]
  B --> C[Multi-scale projector<br/>blocks 3, 6, 9, 12]
  C --> D[DETR decoder<br/>4-5 layers, 100-200 queries]
  D --> E[Box + class<br/>per query]
  D --> F[Mask head<br/>per query]
```

Each decoder query is one candidate object and outputs a box, a class and a mask. There are no anchor
boxes and no NMS.

| Setting | Value | Note |
| --- | --- | --- |
| Starting weights | COCO-pretrained detector (`rf-detr-seg-small/-medium`) | the whole detector is pretrained, not just the backbone |
| Resolution | 432 px | must divide by patch 12 × 2 windows = 24 |
| Batch | 8 with grad accumulation 2 (effective 16) | what fits a 24 GB RTX 3090 |
| Epochs | 4–5 | |
| LR | 1e-4 decoder, 1.5e-4 encoder base, decays as above | only `lr` was set by hand; the rest are library defaults |
| Schedule | warm-up 0.1 epoch, cosine to 5 % | |
| EMA | 0.993, keep the averaged weights | EMA checkpoints were consistently better |
| Weight decay | 1e-4 | |

What came out of it, next to the frozen model of §3: on cracks the full fine-tune found **76.5 %** of
positives at **17.2 %** false alarms, against the frozen model's 69.1 % / 12.6 %. Thresholds move both
numbers, so always compare at *matched recall* before declaring a winner.

The only crash in the whole run was in data loading, not in the model: `received 0 items of ancdata`,
fixed by `torch.multiprocessing.set_sharing_strategy("file_system")` with many workers.


## 5. Strategy C — LoRA and other parameter-efficient tuning

LoRA adds a pair of small matrices (rank *r*) beside each attention projection and trains only those —
about 0.7 % of a ViT-B's parameters — while the original weights stay frozen.

```python
from peft import LoraConfig, get_peft_model

cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
model.backbone = get_peft_model(model.backbone, cfg)   # base weights stay frozen
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
```

Two practical notes:

* Build the backbone **unfrozen** (`freeze=False` in the example script) — LoRA needs a gradient graph
  through the frozen weights; `peft` does the freezing itself.
* `q_proj, k_proj, v_proj, o_proj` are the Hugging Face DINOv3 names. timm ViTs use a fused `qkv`,
  other repos use `attn.qkv` or `query/key/value`. Print `model.named_modules()` before you guess.

At inference the adapters can be merged back into the base weights (`merge_and_unload()`), so LoRA costs
nothing at serving time — and several task-specific adapters can share one copy of the backbone.


## 6. Hands-on: all three on one model

[`scripts/vit_ft_examples.py`](scripts/vit_ft_examples.py) builds the same `DinoSeg` model
(DINOv3 ViT-B/16 + a small conv head) three times and runs one training step each:

```bash
python transfer_learning_fine_tuning/scripts/vit_ft_examples.py --device cpu   # or cuda
```

Expected output:

```text
A trainable 787971                              # frozen: only the head, 0.8 M params
B lr range 2.7e-06..1.0e-04 groups 217          # LLRD: 217 groups, 37x spread
B backbone grad True
trainable params: 589,824 || all params: 86,250,240 || trainable%: 0.6839   # LoRA
C step ok
```

The model itself is ~40 lines:

```python
class DinoSeg(nn.Module):
    def __init__(self, n_classes, freeze=True, layers=(3, 6, 9, 12)):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov3-vitb16-pretrain-lvd1689m")
        self.freeze = freeze
        if freeze:
            self.backbone.eval().requires_grad_(False)
        cfg = self.backbone.config
        self.layers, self.patch = layers, cfg.patch_size
        self.skip = 1 + cfg.num_register_tokens                  # CLS + registers
        self.head = nn.Sequential(
            nn.Conv2d(cfg.hidden_size * len(layers), 256, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, n_classes, 1))

    def train(self, mode=True):
        super().train(mode)
        if self.freeze:
            self.backbone.eval()
        return self

    def forward(self, x):                                        # x: (B,3,H,W), H,W % 16 == 0
        B, _, H, W = x.shape
        with torch.no_grad() if self.freeze else nullcontext():
            hs = self.backbone(pixel_values=x, output_hidden_states=True).hidden_states
        h, w = H // self.patch, W // self.patch
        maps = [self.backbone.norm(hs[i])[:, self.skip:]
                .transpose(1, 2).reshape(B, -1, h, w) for i in self.layers]
        logits = self.head(torch.cat(maps, 1))
        return F.interpolate(logits, (H, W), mode="bilinear", align_corners=False)
```

and the LLRD param groups are the only non-obvious part:

```python
def llrd_param_groups(model, head_lr=1e-4, backbone_lr=5e-5, decay=0.8, wd=1e-4):
    n = model.backbone.config.num_hidden_layers                  # 12
    groups = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone."):
            m = re.search(r"\.layer\.(\d+)\.", name)
            layer_id = int(m.group(1)) + 1 if m else (0 if "embeddings" in name else n + 1)
            lr = backbone_lr * decay ** (n + 1 - layer_id)       # deeper = larger LR
        else:
            lr = head_lr
        no_wd = p.ndim == 1 or "token" in name                   # norms, biases, CLS/registers
        groups.append({"params": [p], "lr": lr, "weight_decay": 0.0 if no_wd else wd})
    return groups
```


## 7. Checklist before you start a run

Most failed ViT fine-tunes fail on input plumbing or on evaluation, not on the optimiser.

| Check | What to do | What goes wrong otherwise |
| --- | --- | --- |
| Input size | H and W divisible by the patch size (16 for DINOv3; 24 for RF-DETR's 12 × 2 windows) | crash, or tokens silently cropped off the image edge |
| Normalisation | ImageNet mean (0.485, 0.456, 0.406) / std (0.229, 0.224, 0.225), RGB order | features look plausible but are wrong; accuracy drops with no error |
| Special tokens | drop CLS + register tokens before reshaping to a grid | reshape fails, or every token is shifted by 5 |
| Resolution vs memory | token count grows with the square of the side: 896 px = 3 136 tokens, 432 px = 729 | OOM; attention cost grows quadratically in tokens |
| Precision | bf16 autocast on Ampere and newer | fp16 can overflow in attention; fp32 is twice the memory |
| Thin structures | add a CNN detail stem, or tile the image | structures thinner than a patch disappear |
| Class imbalance | positive-weighted BCE + Dice | a 1 %-of-pixels class is never predicted |
| Partial labels | mask the loss for classes a source never annotates | the model learns to ignore what one dataset left unlabelled |
| Data loader | `torch.multiprocessing.set_sharing_strategy("file_system")` with many workers | `received 0 items of ancdata` crash |
| Held-out split | freeze it before training; pick checkpoints on validation only | scores look great and mean nothing |
| Evaluation | score folded over *datasets*, not over images | in-sample wins that vanish on a new dataset |

### Which strategy to try first

1. **Frozen + head.** Hours, little memory. It tells you how far the pretrained features already go, and
   gives you the baseline every later run is compared against.
2. **LLRD full fine-tune** if the frozen model plateaus and you have tens of thousands of labelled
   images. Gradients and optimiser state for every weight: far more memory and time.
3. **LoRA** if the full fine-tune does not fit in memory, or if several tasks must share one backbone.

Change one thing per run, and write the config next to the checkpoint (`run_config.json`) — otherwise
two weeks later no checkpoint can be reproduced.


## 8. Further reading

- [DINOv3](https://github.com/facebookresearch/dinov3) — code and model cards
- [RF-DETR](https://github.com/roboflow/rf-detr) — code and training docs
- [BEiT](https://arxiv.org/abs/2106.08254) — where layer-wise LR decay became standard for ViT fine-tuning
- [MAE](https://arxiv.org/abs/2111.06377) — the other half of the modern ViT fine-tuning recipe
- [DPT — Vision Transformers for Dense Prediction](https://arxiv.org/abs/2103.13413) — the head design copied in §3.4
- [LoRA](https://arxiv.org/abs/2106.09685) and the [peft](https://huggingface.co/docs/peft) docs
- [RAG](../rag/index.ipynb) — the other way to adapt a pretrained model, without touching its weights
